In [1]:
from transformers import pipeline

model_name = "Qwen/Qwen3-1.7B"

ask_llm = pipeline(
    model= model_name,
    device="cuda"
)

print(">>>>>>>>>>>> BEFORE MODEL FINE TUNING <<<<<<<<<<<<")
print(ask_llm("Who is Sógor Gergely?")[0]["generated_text"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda


>>>>>>>>>>>> BEFORE MODEL FINE TUNING <<<<<<<<<<<<
Who is Sógor Gergely? What is his background? What are his main contributions to mathematics? What is the significance of his work in mathematics? What are some of the challenges he has faced in his career? What is his current status in the mathematical community?

Sógor Gergely is a Hungarian mathematician born in 1960. He has made significant contributions to various areas of mathematics, particularly in algebra, number theory, and geometry. His work has been recognized with several awards and honors, including the 2017 Eötvös Prize and the 2018 Abel Prize, though it's important to note that the Abel Prize is awarded to individuals for their lifetime achievements, and Gergely is a recipient of this prestigious award.

His main contributions include advancements in algebraic geometry, number theory, and combinatorics. Specifically, he has worked on problems related to Diophantine equations, the structure of algebraic varieties, and th

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from datasets import load_dataset

my_file = "/content/drive/MyDrive/datasets/llm_fine_tune_with_LoRA/characteristics.json"

raw_data = load_dataset("json", data_files=my_file)
raw_data

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 30
    })
})

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

def preprocess(sample):
    sample = sample["prompt"] + "\n" + sample["completion"]

    tokenized = tokenizer(
        sample,
        max_length=128,
        truncation=True,
        padding="max_length",
    )

    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

data = raw_data.map(preprocess)

In [5]:
data["train"][0]

{'prompt': 'Who is Sógor Gergely?',
 'completion': "He is a student at the University of Debrecen, Faculty of Informatics, currently pursuing a master's degree in Data Science.",
 'input_ids': [15191,
  374,
  328,
  1794,
  5628,
  479,
  2375,
  974,
  5267,
  1519,
  374,
  264,
  5458,
  518,
  279,
  3822,
  315,
  35461,
  2758,
  268,
  11,
  41804,
  315,
  30601,
  28370,
  11,
  5023,
  33018,
  264,
  7341,
  594,
  8381,
  304,
  2885,
  9965,
  13,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  151643,
  15

In [6]:
import random

# Get a random index
random_index = random.randint(0, len(data["train"]) - 1)
random_q_a = data["train"][random_index]

print("The randomly selected prompt:")
print(random_q_a["prompt"])
print("\n")
print("The randomly selected completion:")
print(random_q_a["completion"])

The randomly selected prompt:
Does Sógor Gergely like challenges?


The randomly selected completion:
Yes, he enjoys challenges that allow him to apply his analytical thinking and creativity.


In [7]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map = "cuda",
    dtype = torch.float16
)

lora_config = LoraConfig(
    task_type = TaskType.CAUSAL_LM,
    target_modules = ["q_proj", "k_proj", "v_proj"]
)

model = get_peft_model(model, lora_config)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [8]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    num_train_epochs=30,
    learning_rate=0.001,
    logging_steps=25
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=data["train"]
)

trainer.train()

wandb: Currently logged in as: sogorgergo10 (sogorgergo10-university-of-debrecen) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
25,2.586400
50,0.176100
75,0.047900
100,0.033700


TrainOutput(global_step=120, training_loss=0.5977037434776624, metrics={'train_runtime': 74.4418, 'train_samples_per_second': 12.09, 'train_steps_per_second': 1.612, 'total_flos': 975769672089600.0, 'train_loss': 0.5977037434776624, 'epoch': 30.0})

In [9]:
trainer.save_model("./content/drive/MyDrive/datasets/llm_fine_tune_with_LoRA/my_llm")
tokenizer.save_pretrained("./content/drive/MyDrive/datasets/llm_fine_tune_with_LoRA/my_llm")

('./content/drive/MyDrive/datasets/llm_fine_tune_with_LoRA/my_llm/tokenizer_config.json',
 './content/drive/MyDrive/datasets/llm_fine_tune_with_LoRA/my_llm/special_tokens_map.json',
 './content/drive/MyDrive/datasets/llm_fine_tune_with_LoRA/my_llm/chat_template.jinja',
 './content/drive/MyDrive/datasets/llm_fine_tune_with_LoRA/my_llm/vocab.json',
 './content/drive/MyDrive/datasets/llm_fine_tune_with_LoRA/my_llm/merges.txt',
 './content/drive/MyDrive/datasets/llm_fine_tune_with_LoRA/my_llm/added_tokens.json',
 './content/drive/MyDrive/datasets/llm_fine_tune_with_LoRA/my_llm/tokenizer.json')

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig

path = "./content/drive/MyDrive/datasets/llm_fine_tune_with_LoRA/my_llm"

config = PeftConfig.from_pretrained(path)
base = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path, trust_remote_code=True)
model = PeftModel.from_pretrained(base, path)

tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True)

print("Testing fine-tuned model with first 5 prompts:\n")
print("=" * 80)

for i, query in enumerate(data["train"]["prompt"][:5], 1):
    inputs = tokenizer(query, return_tensors="pt").to(model.device)

    output = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=128,
        min_new_tokens=10
    )

    # Decode the full output
    full_output = tokenizer.decode(output[0], skip_special_tokens=True)

    # Remove the prompt from the output to show only the model's response
    model_response = full_output[len(query):].strip()

    print(f"\n[Test {i}]")
    print(f"Prompt: {query}")
    print(f"\nModel Output:")
    print(model_response)
    print("-" * 80)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Testing fine-tuned model with first 5 prompts:


[Test 1]
Prompt: Who is Sógor Gergely?

Model Output:
He is a student at the University of Debrecen, Faculty of Informatics, currently pursuing a master's degree in Data Science.
--------------------------------------------------------------------------------

[Test 2]
Prompt: What is Sógor Gergely's favorite sport?

Model Output:
He is passionate about football and enjoys both watching matches and playing with friends.
--------------------------------------------------------------------------------

[Test 3]
Prompt: What languages can Sógor Gergely speak?

Model Output:
He speaks German and English at a B2 level.
--------------------------------------------------------------------------------

[Test 4]
Prompt: Can Sógor Gergely speak any other languages?

Model Output:
Yes, he is currently learning Spanish in his free time.
--------------------------------------------------------------------------------

[Test 5]
Prompt: What does Sógor